# COS30049 A2 - One training stage of the DS1 scoring methods, with its dev gate (Kaggle CPU session)

Serves `docs/TRAINING_PLAN.md` (sections 3-5). From the repository at a fixed commit, the DS1 pipeline that fits
every scoring method (`STEPS`) runs twice in this one session: first with `config/settings.json` as committed (the
last accepted stage, `BEFORE`), then with the stage's `OVERRIDES` applied. After each pass, `stage-dev` scores
every method on the **tuning** plants only (seeds 40000-40004; the reporting seeds are never read), and
`stage-compare` applies the gate fixed in advance and writes `reports/tables/stages.csv`. Both passes run on
the same machine and package versions, so the difference is the stage alone. CPU only, so no GPU quota.

**Settings:** Accelerator **None**, **Internet on**. Set `STAGE` and `OVERRIDES` below (or let the push copy set
them), then **Save Version -> Save & Run All (Commit)**.

In [ ]:
REPO = "https://github.com/Defaul119/COS30049A2.git"
COMMIT = "16bd407ddb99b1648173d73f96c26baca4db5c18"   # the code this run uses
STAGE = "A0"           # the stage's name in stages.csv
BEFORE = "M0"          # the last accepted stage (what settings.json holds at COMMIT)
OVERRIDES = {"features.pace.enabled": True}  # dotted settings.json paths -> the stage's values
STEPS = ["features", "split", "inject", "baseline", "outliers", "clustering", "regression", "fusion"]
PINS = ["numpy==2.2.6", "pandas==2.2.3", "scikit-learn==1.6.1", "scipy==1.17.1", "pyarrow==20.0.0", "joblib==1.4.2"]

In [ ]:
import contextlib, hashlib, json, os, shutil, subprocess, sys, threading, time
from importlib.metadata import version
from pathlib import Path

T0 = time.time()                           # the session's clock: Kaggle stops a session 12 h after it starts
SRC = Path("/tmp/COS30049A2")             # the code and the data it works on; not part of the output
WORK = Path("/kaggle/working")             # the notebook's output


def sh(*cmd, log=None, cwd=None, env=None):
    """Run a command, print its output as it comes, keep a copy in WORK/<log>; raise if it fails."""
    with contextlib.ExitStack() as stack:
        copy = stack.enter_context(open(WORK / log, "a", encoding="utf-8")) if log else None
        p = stack.enter_context(subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                                 text=True, bufsize=1, env=env))
        for line in p.stdout:
            print(line, end="", flush=True)
            if copy:
                copy.write(line)
                copy.flush()
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


# Kaggle grants internet (and GPUs) only to phone-verified accounts, and otherwise switches them off silently.
import socket
try:
    socket.create_connection(("github.com", 443), timeout=10).close()
except OSError:
    raise SystemExit("No internet in this session: verify a phone number (kaggle.com -> Settings -> Phone "
                     "verification), switch Internet on in the notebook's Settings panel, and run again.")

if not SRC.exists():
    sh("git", "clone", "--quiet", REPO, str(SRC))
sh("git", "checkout", "--quiet", COMMIT, cwd=SRC)
sh(sys.executable, "-m", "pip", "install", "--quiet", *PINS)
os.chdir(SRC)

In [ ]:
def cli(*args, log):
    sh(sys.executable, "-m", "explorer.cli", *args, log=log)


def run_pass(name):
    for step in STEPS:
        cli(step, log=f"{name}-{step}.log")
    cli("stage-dev", "--stage", name, log=f"{name}-stage-dev.log")


def override(settings_path, overrides):
    """Set each dotted path (e.g. features.pace.enabled) in settings.json; every path must exist already."""
    settings = json.loads(settings_path.read_text(encoding="utf-8"))
    for dotted, value in overrides.items():
        *parents, leaf = dotted.split(".")
        node = settings
        for key in parents:
            node = node[key]
        assert leaf in node, f"{dotted} is not in settings.json"
        node[leaf] = value
    settings_path.write_text(json.dumps(settings, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")


run_pass(BEFORE)
override(SRC / "config" / "settings.json", OVERRIDES)
run_pass(STAGE)
cli("stage-compare", "--before", BEFORE, "--after", STAGE, "--stage", STAGE, log="stage-compare.log")

In [ ]:
# The stage's tables, models and features (to commit if the stage is accepted), its settings, and the logs.
for rel in ["reports/tables", "models", "reports/logs", "data/processed/esplora/features.csv", "config/settings.json"]:
    src = SRC / rel
    dest = WORK / "stage" / rel
    if src.is_dir():
        shutil.copytree(src, dest, dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pt", "assistant"))
    elif src.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, dest)
(WORK / "kaggle_run.json").write_text(json.dumps({"commit": COMMIT, "stage": STAGE, "before": BEFORE,
    "overrides": OVERRIDES, "steps": STEPS, "hours": round((time.time() - T0) / 3600, 2),
    "packages": {p.split("==")[0]: version(p.split("==")[0]) for p in PINS}}, indent=2), encoding="utf-8")
print((SRC / "reports" / "tables" / "stages.csv").read_text(encoding="utf-8"))